# Experiment A (fast) — LSTM / BiLSTM soil-moisture forecast, 48h horizon (FD11, 2015–2024)

Forecasts `SWC_20` (soil water content at 20 cm) at station `FD11`, **48 hours** after the end of a 24-hour
input window, from two inputs only — `SWC_20` itself and the 24-hour rainfall total — with a small unidirectional
or bidirectional LSTM (the retuning grid search picks one), and compares it against naive baselines (last value,
24h moving average).

**Fast version of `experiment_a_48h.ipynb`:** the same data, split, grid and final model, with three changes. There
is no feature selection: the inputs are fixed to `SWC_20` and `Ppt_rolling_24h_log`. Cross-validated fits stop
after `CV_PATIENCE` = 3 epochs without validation improvement, at most `TUNING_EPOCHS` = 15 (the final model keeps
30 epochs / patience 10). Training batches are sliced from tensors kept on the GPU instead of being collated and
copied one at a time (same batches, same weights).

**Pipeline:** data load → preprocessing → year-based split → retune (LSTM vs BiLSTM) on whole-year CV folds →
final train → evaluation against the baselines.

| | |
|---|---|
| Data | `TxSON33_modeling_complete.parquet`, station `FD11`, hourly, 2015-01-01 00:00 to 2024-12-31 23:00 (10 full calendar years) |
| Target | `SWC_20`; only `observed` readings are used as training targets or scored |
| Lookback / horizon | 24 hours / 48 hours: inputs cover hours `[t, t+23]`, the target is `SWC_20` at `t + 23 + 48` |
| Inputs (fixed) | `SWC_20` and `Ppt_rolling_24h_log` (24-hour rainfall total; hourly rain below 1 mm is zeroed first) |
| Network output | the change in `SWC_20` from the window's last hour to the target hour (`PREDICT_CHANGE`), added back to that last value; every score is in `SWC_20` units |
| Split | by calendar year: train 2015–2022 (8 years), validation 2023 (early stopping only), test 2024 (used once, at the end) |
| Retuning | whole-year, expanding-window CV inside the training years: train 2015–2018 → validate 2019–2020; train 2015–2020 → validate 2021–2022 |
| Retune grid | `model_type` {lstm, bilstm} × `num_layers` {1, 2} × `hidden_size` {32, 48} × `lr` {0.001, 0.005}, `batch_size` 16 |
| Training | Adam, MSE loss; CV fits: up to 15 epochs, early stopping patience 3; final model: up to 30 epochs, patience 10 |
| Baselines | last value (persistence) and 24h moving average, scored on exactly the same targets |

In [ ]:
# --- Setup and reproducibility ---
import json
import os
import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display
from joblib import Parallel, delayed
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ParameterGrid

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

# Repo root, one level above this notebook. In Colab, upload the parquet (or mount Drive) and point this at it.
PARQUET_PATH = "../TxSON33_modeling_complete.parquet"
STATION = "FD11"
START_DATE = "2015-01-01"
END_DATE_EXCLUSIVE = "2025-01-01"      # 2015-2024 inclusive: 10 full calendar years
PERIOD_LABEL = f"{START_DATE[:4]}-{int(END_DATE_EXCLUSIVE[:4]) - 1}"   # "2015-2024", for titles and file names
# Year-based split, chronological: training years, then the validation year, then the test year
TRAIN_YEARS = list(range(2015, 2023))  # 2015-2022: CV for retuning, final model fit
VAL_YEARS = [2023]                     # early stopping of the final model only
TEST_YEARS = [2024]                    # touched once, at the end
# Whole-year, expanding-window CV folds inside TRAIN_YEARS: (fold training years, fold validation years)
CV_FOLD_YEARS = [
    (list(range(2015, 2019)), [2019, 2020]),   # fold 1: train 2015-2018, validate 2019-2020
    (list(range(2015, 2021)), [2021, 2022]),   # fold 2: train 2015-2020, validate 2021-2022
]
TARGET_COL = "SWC_20"
FEATURES = ["SWC_20", "Ppt_rolling_24h_log"]   # the model inputs: SWC_20 and the 24-hour rainfall total
PPT_DEADBAND_MM = 1.0  # hourly rainfall below this (mm) is set to 0 before the rainfall total
WINDOW_SIZE = 24                       # hours of past data used as input
HORIZON = 48                           # target = SWC_20 this many hours after the window's last hour
EXPERIMENT_NAME = "Experiment A"
# True: the network predicts the change in SWC_20 from the window's last hour to the target hour, which is added
# back to that last value (Section 5). False: it predicts the SWC_20 level directly, as in earlier runs.
PREDICT_CHANGE = True

# Retuning grid (16 combinations); model_type compares LSTM and BiLSTM
PARAM_GRID = {
    "model_type": ["lstm", "bilstm"],
    "num_layers": [1, 2],
    "hidden_size": [32, 48],
    "lr": [0.001, 0.005],
    "batch_size": [16],
}

EPOCHS = 30            # max epochs for the final model
PATIENCE = 10          # final model early stopping: stop after 10 epochs without validation improvement
TUNING_EPOCHS = 15     # max epochs for every cross-validated fit (retuning)
CV_PATIENCE = 3        # CV fits stop after 3 epochs without validation improvement (speed)
EVAL_BATCH_SIZE = 1024  # batch size for validation losses and predictions only (speed; training batches stay 16)
# CV fits are independent and each is seeded the same way, so they can run in parallel worker processes. On a GPU
# they run one at a time in this process; on a CPU, up to 4 single-threaded workers.
N_JOBS = 1 if DEVICE.type == "cuda" else min(4, os.cpu_count() or 1)

# Plot colors: actual = ink, model = blue, last-value baseline = orange, reference lines = gray
C_ACTUAL, C_MODEL, C_LAST, C_MUTED = "#0b0b0b", "#2a78d6", "#eb6834", "#8a8984"

# Retuning table, test metrics and the run summary are saved here.
# In Colab, point this at a mounted Drive folder (e.g. "/content/drive/MyDrive/txson_outputs") to keep the files.
OUTPUT_DIR = "outputs_experiment_a_48h_fast"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def output_path(name):
    return os.path.join(OUTPUT_DIR, f"{STATION}_{PERIOD_LABEL}_h{HORIZON}_{name}")

## 1. Load and inspect the data

Loading the parquet file, filtering to one station and 2015–2024 (the station's full history is kept aside for
the rolling rainfall features), and checking what that slice contains — including its `*_Origin` composition
(`observed` / `pipeline_imputed` / `modeling_completion`), overall and, for the target and precipitation, by
calendar year. The dataset's README is explicit that `modeling_completion` values are synthetic and should not be
used as validation or test ground truth.

In [ ]:
raw = pd.read_parquet(PARQUET_PATH)
raw["Timestamp"] = pd.to_datetime(raw["Timestamp"])

# Full, chronologically sorted station history -- kept so the rolling rainfall features (Section 2) can use the
# hours before START_DATE -- and the modeling period START_DATE <= Timestamp < END_DATE_EXCLUSIVE
station_df = raw[raw["Station"] == STATION].copy()
station_df = station_df.sort_values("Timestamp").set_index("Timestamp")
df = station_df[(station_df.index >= START_DATE) & (station_df.index < END_DATE_EXCLUSIVE)].copy()

RAW_INPUT_COLS = ["SWC_20", "Ppt"]
ORIGINS = ["observed", "pipeline_imputed", "modeling_completion"]
expected_rows = len(pd.date_range(START_DATE, END_DATE_EXCLUSIVE, freq="h", inclusive="left"))

print("Shape:", df.shape, f"(a complete hourly {PERIOD_LABEL} has {expected_rows} rows)")
print("Date range:", df.index.min(), "to", df.index.max())
print("Duplicate timestamps:", df.index.duplicated().sum())
print("Hourly and gap-free:", (df.index.to_series().diff().dropna() == pd.Timedelta("1h")).all())
print(f"Full {STATION} history (for rolling features):", station_df.index.min(), "to", station_df.index.max())
print()
print("Missing values in the columns used below:")
print(df[RAW_INPUT_COLS].isna().sum().to_dict())
print()
print("Origin composition of every raw input column (modeling_completion = synthetic):")
print(pd.DataFrame({c: df[f"{c}_Origin"].value_counts() for c in RAW_INPUT_COLS}).T
      .reindex(columns=ORIGINS).fillna(0).astype(int))
print()
print("Target and precipitation origin by calendar year (every year must be observed data, not synthetic):")
origin_by_year = pd.concat({c: pd.crosstab(df.index.year, df[f"{c}_Origin"]).reindex(columns=ORIGINS, fill_value=0)
                            for c in ["SWC_20", "Ppt"]}, axis=1)
origin_by_year.index.name = "year"
print(origin_by_year.to_string())
print()
print("Hours where the target is not an observed reading:",
      [str(t) for t in df.index[df["SWC_20_Origin"] != "observed"]])

**What this shows:** 87,672 hourly rows for 2015–2024 with no missing values and no duplicate or missing
timestamps. The station's history runs from 2014-11-12 to 2025-07-16, so 2015–2024 is the only window of 10
complete calendar years. By year, `SWC_20` is `observed` for every hour except 4 `pipeline_imputed` ones — two in
2016 (2016-03-14 00:00, 2016-11-07 01:00) and two in 2019 (2019-10-24 16:00, 2019-11-18 12:00) — and `Ppt` for
every hour except 2 (2016-03-14 00:00, 2019-10-24 16:00). Every other year is 100% `observed` for both, and
neither column has a single `modeling_completion` (synthetic) value in any year. All four non-observed target
hours fall in the training years; the validation (2023) and test (2024) years are fully observed. Any window
whose target hour is not an `observed` `SWC_20` reading is dropped from training, validation and test scoring.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(df.index, df["SWC_20"], linewidth=0.6, color=C_MODEL)
axes[0].set_ylabel("SWC_20")
axes[0].set_title(f"{STATION}, {PERIOD_LABEL}: soil moisture at 20 cm (target) and hourly precipitation")
axes[1].plot(df.index, df["Ppt"], linewidth=0.6, color=C_ACTUAL)
axes[1].set_ylabel("Ppt")
axes[1].set_xlabel("Date")
plt.tight_layout()
plt.show()

`SWC_20` shows a slow-drydown-with-sharp-rises pattern: soil moisture creeps down between rain events and jumps
when it rains. The drydown depends on how wet the soil has been over the preceding hours, and the rises line up
with precipitation — which is why rainfall is the second input.

## 2. Preprocessing and feature engineering

### Precipitation: deadband

`Ppt` comes from a tipping-bucket gauge (one tip = 0.254 mm). Hourly readings below `PPT_DEADBAND_MM` (1 mm, i.e.
fewer than four tips in the hour) are set to zero before the rainfall total is computed. Rain that light is held
by the canopy, litter and the top few centimetres of soil and evaporates without reaching 20 cm, so it should not
move a 20 cm forecast; a network fed every single tip can learn to forecast jumps that never happen. Over
2015–2024 the deadband zeroes 58% of rainy hours but only 10.6% of the rain volume. Negative readings are
clipped to zero first as a defensive step (`log1p` is undefined below -1); there are none in this period.

### 24-hour rainfall total (the rain input)

`Ppt_rolling_24h_log` is `log1p` of the deadbanded rainfall total over the 24 hours **ending at** each hour (the
current hour included). A 24-hour total barely moves with one tip but tracks the accumulated depth of a storm,
which is what drives infiltration to 20 cm; over the training years its correlation with `SWC_20` (0.27) is
about three times that of hourly rain (0.095). Across the 24-hour input window, the 24-hour totals also carry
rain from up to 47 hours before the window ends, so a storm that began before the window is still visible.

The total is computed on the full, chronologically sorted `FD11` history and then attached to the 2015–2024
rows, so the first hours of 2015 are initialized from real late-2014 rainfall. A missing hour is never counted as
zero rain: each sum needs every hour of its window (`min_periods` = window length, on a complete hourly grid), so
an incomplete window stays `NaN` and is trimmed from the start of the period (none are expected; the history
begins 2014-11-12).

In [ ]:
n_negative = int((df["Ppt"] < 0).sum())
print(f"Negative Ppt readings found: {n_negative} out of {len(df)}")
df["Ppt"] = df["Ppt"].fillna(0.0).clip(lower=0.0)   # defensive; no-op for this station/period


def deadband(ppt, threshold=PPT_DEADBAND_MM):
    """Hourly rainfall below `threshold` mm set to 0 (too light to infiltrate to 20 cm). NaN stays NaN."""
    return ppt.mask(ppt < threshold, 0.0)


rainy = df["Ppt"] > 0
zeroed = rainy & (df["Ppt"] < PPT_DEADBAND_MM)
print(f"Deadband (< {PPT_DEADBAND_MM} mm/h): zeroes {zeroed.sum() / rainy.sum():.1%} of rainy hours, "
      f"{df.loc[zeroed, 'Ppt'].sum() / df['Ppt'].sum():.1%} of the rain volume")


def rolling_ppt_log(ppt, hours):
    """log1p of the total rainfall over the `hours` hours ENDING AT each timestamp (the current hour included).

    ppt: chronologically sorted hourly series. It is put on a complete hourly grid first, so a missing row or a
    NaN reading makes that window's sum NaN (min_periods = hours) instead of silently counting as zero rain.
    """
    ppt = ppt.asfreq("h")
    return np.log1p(ppt.rolling(hours, min_periods=hours).sum())


# Computed on the full station history (negatives clipped and the deadband applied as above, missing values left
# as NaN) and then assigned to the modeling-period rows, aligned on Timestamp
history_ppt = deadband(station_df["Ppt"].clip(lower=0.0))
df["Ppt_rolling_24h_log"] = rolling_ppt_log(history_ppt, 24)

# Explicit warm-up handling: NaN means part of that hour's rolling window is missing from the history
incomplete = df["Ppt_rolling_24h_log"].isna()
print(f"Rows whose rolling-rainfall window is incomplete: {int(incomplete.sum())} of {len(df)}")
if incomplete.any():
    first_complete = incomplete.idxmin()
    assert not incomplete.loc[first_complete:].any(), "incomplete rolling windows after the warm-up; check Ppt gaps"
    print(f"Dropping {int(incomplete.loc[:first_complete].sum())} leading warm-up rows; "
          f"modeling starts at {first_complete}")
    df = df.loc[first_complete:].copy()
assert not df[FEATURES].isna().any().any(), "a model input contains NaN"

## 3. Year-based train / validation / test split (2015–2022 / 2023 / 2024)

Whole calendar years, chronological: the training years come entirely before the validation year, which comes
entirely before the test year. Every partition covers complete years, so every season appears in each of them.

- **Train, 2015–2022 (8 years)** — fits the model. Retuning runs on whole-year cross-validation folds
  *inside* this block only (Section 10).
- **Validation, 2023** — used only for early stopping of the final model.
- **Test, 2024** — touched once, at the very end, to compare the model against the naive baselines.

In [ ]:
def rows_in_years(data_df, years):
    """Rows of data_df whose timestamp falls in one of the calendar `years`."""
    return data_df[data_df.index.year.isin(years)]


def year_span(years):
    """'2015-2018' for a run of years, '2023' for a single year."""
    return f"{years[0]}-{years[-1]}" if len(years) > 1 else str(years[0])


def assert_whole_years(part, years):
    """part must be exactly the consecutive calendar `years`, every hour present."""
    assert list(years) == list(range(years[0], years[-1] + 1)), f"{years} are not consecutive years"
    full = pd.date_range(f"{years[0]}-01-01", f"{years[-1] + 1}-01-01", freq="h", inclusive="left")
    assert len(part) == len(full) and (part.index == full).all(), f"{year_span(years)} is not complete hourly years"


train_df = rows_in_years(df, TRAIN_YEARS)
val_df = rows_in_years(df, VAL_YEARS)
test_df = rows_in_years(df, TEST_YEARS)

for name, part, years in [("train", train_df, TRAIN_YEARS), ("val", val_df, VAL_YEARS), ("test", test_df, TEST_YEARS)]:
    assert_whole_years(part, years)
    print(f"{name:5s}: {year_span(years):9s} ({len(years)} year{'s' if len(years) > 1 else ''}) {len(part):6d} rows  "
          f"{part.index.min()} -> {part.index.max()}")
assert len(train_df) + len(val_df) + len(test_df) == len(df)
assert train_df.index.max() < val_df.index.min() <= val_df.index.max() < test_df.index.min()

print()
print("Validation and test SWC_20 origin composition (must be observed ground truth, not synthetic):")
for name, part in [("val", val_df), ("test", test_df)]:
    print(f"  {name}: {part['SWC_20_Origin'].value_counts().to_dict()}")

## 4. Normalization

`MinMaxScaler`, fit on the training split only and reused (never refit) for validation and test, so their
distributions never leak into preprocessing. Validation/test values can legitimately fall outside `[0, 1]`.

In [ ]:
scaler = MinMaxScaler()
scaler.fit(train_df[FEATURES])   # fit on TRAIN ONLY; per-column, so any feature subset is scaled the same

def scale(split_df, cols=FEATURES, fitted_scaler=scaler):
    arr = fitted_scaler.transform(split_df[cols]).astype(np.float32)
    return pd.DataFrame(arr, columns=cols, index=split_df.index)


def unscale_target(values, fitted_scaler=scaler):
    """Scaled SWC_20 values back to original SWC_20 units."""
    pos = list(fitted_scaler.feature_names_in_).index(TARGET_COL)
    lo, hi = fitted_scaler.data_min_[pos], fitted_scaler.data_max_[pos]
    return np.asarray(values, dtype=np.float64) * (hi - lo) + lo


train_scaled = scale(train_df)
val_scaled = scale(val_df)
test_scaled = scale(test_df)
assert np.allclose(train_scaled.min(), 0) and np.allclose(train_scaled.max(), 1)   # train is exactly [0, 1]

## 5. Building X/Y windows

A 24-hour window slides across the hourly data; each position is one example `X`, and its label `y` is `SWC_20`
exactly `HORIZON` hours after the window's last hour: inputs at rows `[start, end - 1]`, target at
`end - 1 + HORIZON`. Written as an explicit loop so every step is visible.

Windows are built **separately inside each partition** (train / validation / test, and each CV fold's two parts),
so an input window and its target never straddle a boundary. Windows whose target hour is not an `observed`
`SWC_20` reading are dropped after windowing (never the source rows), so every kept window still ends exactly
`HORIZON` hours before its target.

**Change target (`PREDICT_CHANGE`).** `make_windows` returns the scaled `SWC_20` *level* at the target hour;
`observed_windows` turns it into the scaled change since the window's last hour,
`y = SWC_20(target hour) − SWC_20(last input hour)`, and `to_level` adds the network's predicted change back to
that last value before unscaling. `SWC_20` moves slowly, so with a level target the easiest way for the network
to lower MSE is to reproduce its last input value — a forecast that follows the actual series shifted by the
horizon, i.e. persistence, plus whatever bias it has in reproducing the level. With a change target, an output of
0 *is* persistence, and everything the network learns goes into what persistence misses (the drydown between rain
events, the 20 cm response to rain already in the window). The squared error of the change equals the squared
error of the level, so losses and RMSEs are directly comparable with level-target runs. Rain that falls after the
input window cannot be anticipated from past inputs, so rises caused by it are still forecast late; that part of
the error needs a rainfall forecast, not a better network.

In [ ]:
def make_windows(data_df, feature_cols, target_col, window_size, horizon):
    """
    data_df: chronologically ordered, already-scaled dataframe.
    window_size: number of past hours used as input (X).
    horizon: how many hours AFTER the window's last hour the target sits.

    Returns X (num_samples, window_size, num_features), y (num_samples,), and the row-index
    (within data_df) of each window's last input hour / its target hour, for verification.
    """
    values = data_df[feature_cols].values
    targets = data_df[target_col].values
    n = len(data_df)

    X, y, end_idx, target_idx = [], [], [], []
    for start in range(0, n - window_size - horizon + 1):
        end = start + window_size          # first index AFTER the window (exclusive)
        tgt = end - 1 + horizon            # horizon hours after the window's LAST hour (end - 1)
        X.append(values[start:end])
        y.append(targets[tgt])
        end_idx.append(end - 1)
        target_idx.append(tgt)

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32), end_idx, target_idx


def check_window_alignment(index, end_idx, target_idx, window_size, horizon):
    """Asserts every input covers `window_size` consecutive hourly steps and every target is exactly `horizon`
    hours after its window's last input hour, all inside the partition that `index` belongs to."""
    index = pd.DatetimeIndex(index)
    end_idx, target_idx = np.asarray(end_idx), np.asarray(target_idx)
    assert len(target_idx) > 0, "partition too short for one window"
    assert (np.diff(index.values) == np.timedelta64(1, "h")).all(), "partition is not consecutive hourly data"
    start_idx = end_idx - (window_size - 1)
    assert start_idx.min() >= 0 and target_idx.max() < len(index)       # nothing outside this partition
    assert ((index[end_idx] - index[start_idx]) == pd.Timedelta(hours=window_size - 1)).all()
    assert ((index[target_idx] - index[end_idx]) == pd.Timedelta(hours=horizon)).all()


def observed_target_mask(split_df, target_idx):
    """True for windows whose target hour has an observed SWC_20 reading. pipeline_imputed and synthetic
    modeling_completion values are never used as training targets or as validation/test ground truth.
    split_df: the unscaled partition the windows were built from (it carries SWC_20_Origin)."""
    origin = split_df[f"{TARGET_COL}_Origin"].to_numpy()
    return origin[np.asarray(target_idx, dtype=int)] == "observed"


def last_scaled_target(X, feature_cols):
    """Scaled SWC_20 at each window's last input hour (SWC_20 is always an input)."""
    return X[:, -1, list(feature_cols).index(TARGET_COL)]


def observed_windows(split_df, split_scaled, feature_cols):
    """Windows of one partition whose target is an observed SWC_20 reading: X, y (scaled), end_idx, target_idx.
    With PREDICT_CHANGE, y is the scaled change from the window's last hour to the target hour."""
    X, y, end_idx, target_idx = make_windows(split_scaled, feature_cols, TARGET_COL, WINDOW_SIZE, HORIZON)
    keep = observed_target_mask(split_df, target_idx)
    X, y = X[keep], y[keep]
    if PREDICT_CHANGE:
        y = y - last_scaled_target(X, feature_cols)
    return X, y, np.asarray(end_idx)[keep], np.asarray(target_idx)[keep]


def to_level(X, pred, feature_cols, fitted_scaler):
    """Network outputs for windows X -> SWC_20 predictions in original units. With PREDICT_CHANGE the predicted
    change is first added back to the scaled SWC_20 at each window's last input hour."""
    if PREDICT_CHANGE:
        pred = pred + last_scaled_target(X, feature_cols)
    return unscale_target(pred, fitted_scaler)

In [ ]:
# Worked example with real timestamps: first, middle and last training window. The scaled target, converted
# back, must equal the SWC_20 value looked up by the target's TIMESTAMP (independent of the index arithmetic).
X_demo, y_demo, end_idx_demo, target_idx_demo = make_windows(
    train_scaled, FEATURES, TARGET_COL, WINDOW_SIZE, HORIZON)
for i in (0, len(y_demo) // 2, len(y_demo) - 1):
    t_start = train_df.index[end_idx_demo[i] - (WINDOW_SIZE - 1)]
    t_end, t_tgt = train_df.index[end_idx_demo[i]], train_df.index[target_idx_demo[i]]
    y_orig = unscale_target(y_demo[i])
    print(f"window {i:5d}: inputs {t_start} -> {t_end} | target {t_tgt} "
          f"(+{(t_tgt - t_end) / pd.Timedelta(hours=1):.0f}h) | y = {y_orig:.4f}, "
          f"SWC_20 at {t_tgt} = {df.loc[t_tgt, TARGET_COL]:.4f}")
    assert t_tgt - t_end == pd.Timedelta(hours=HORIZON)
    assert abs(y_orig - df.loc[t_tgt, TARGET_COL]) < 1e-6
print(f"X dims: {X_demo.shape} = (num_samples, window_size, num_features);  y dims: {y_demo.shape}")

# The training target round-trips: to_level(X, y) gives back the observed SWC_20 at every target hour
X_obs, y_obs, _, target_idx_obs = observed_windows(train_df, train_scaled, FEATURES)
assert np.allclose(to_level(X_obs, y_obs, FEATURES, scaler),
                   train_df[TARGET_COL].to_numpy()[target_idx_obs], atol=1e-5)

# Every window of every partition: 24 consecutive hourly inputs, target exactly HORIZON hours after the last input,
# never across a partition boundary; plus the count of windows dropped for a non-observed target
exclusions = {}
for part_name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    _, _, end_idx, target_idx = make_windows(part, [TARGET_COL], TARGET_COL, WINDOW_SIZE, HORIZON)
    check_window_alignment(part.index, end_idx, target_idx, WINDOW_SIZE, HORIZON)
    keep = observed_target_mask(part, target_idx)
    exclusions[part_name] = {"windows": len(keep), "observed_targets": int(keep.sum()),
                             "non_observed_targets_dropped": int((~keep).sum())}
print(f"Window alignment checks passed for the {HORIZON}h horizon in train, val and test.")
pd.DataFrame(exclusions).T

## 6. Two LSTM variants: unidirectional (`SimpleLSTM`) and bidirectional (`SimpleBiLSTM`)

Both read the 24-hour input sequence with an `nn.LSTM` (`batch_first=True`) and map the LSTM's final summary
through one `nn.Linear` layer to a single value (structure after the GeeksforGeeks PyTorch LSTM tutorial,
[Long Short-Term Memory Networks using PyTorch](https://www.geeksforgeeks.org/deep-learning/long-short-term-memory-networks-using-pytorch/)).
They share the constructor `(num_features, hidden_size, num_layers)` and both return `(batch, 1)`;
`config["model_type"]` (`"lstm"` or `"bilstm"`) selects one in `build_model`, so the retuning grid and the final
model go through the same code path.

```
input         (batch, 24 hours, F features)   <- SWC_20, Ppt_rolling_24h_log

SimpleLSTM    nn.LSTM(input_size=F, hidden_size=H, num_layers=L, batch_first=True)
                reads hour 1 -> hour 24 -> h_n[-1], the last layer's final hidden state   (batch, H)
              Linear(H -> 1)

SimpleBiLSTM  nn.LSTM(input_size=F, hidden_size=H, num_layers=L, batch_first=True, bidirectional=True)
                forward direction  reads hour 1 -> hour 24 -> h_n[-2]                    (batch, H)
                backward direction reads hour 24 -> hour 1 -> h_n[-1]                    (batch, H)
              concat [h_fwd, h_bwd]  (batch, 2H)  ->  Linear(2H -> 1)

output        (batch, 1)   <- scaled change in SWC_20 from the window's last hour to HORIZON hours later
```

- **LSTM cell:** at every hour the forget, input and output gates update a cell state (long-term memory) and a
  hidden state, which lets the model carry "it rained 10 hours ago" or "the soil has been drying steadily"
  forward through the window, in order.
- **Unidirectional vs bidirectional:** the unidirectional LSTM is the standard choice for forecasting. A BiLSTM
  layer runs two LSTMs, so at the same `hidden_size` it has roughly twice the parameters (more with 2 layers,
  since its second layer reads a 2H-wide input). The retuning grid compares the two on the same CV folds.
- **`bidirectional=True` is not leakage:** the backward direction only reads hours *inside* the already-observed
  input window, never the target hour, which is `HORIZON` hours after the window ends.
- **Why `h_n` rather than `out[:, -1, :]`:** for a bidirectional LSTM, `out[:, -1, :]` holds the backward state
  after only one hour (the backward pass *starts* at hour 24). `h_n[-2]` (forward) and `h_n[-1]` (backward) are
  each direction's final state after reading all 24 hours, always from the last layer. For the unidirectional
  LSTM, `h_n[-1]` is the last layer's state after hour 24.
- **Initial states:** PyTorch uses zero `h0`/`c0` when none are passed; each window starts from a blank memory.

| Setting | Value | Notes |
|---|---|---|
| `model_type` | {`lstm`, `bilstm`} | 16-combination retuning grid |
| `num_layers` | {1, 2} | |
| `hidden_size` | {32, 48} | per direction |
| `lr` | {0.001, 0.005} | Adam |
| `batch_size` | 16 | fixed |
| window | 24 hours | fixed |
| loss | MSE | on the scaled change; minimizing it also minimizes RMSE, the retuning score |
| epochs | CV fits: up to 15, patience 3; final model: up to 30, patience 10 | the best-validation epoch's weights are restored |

Related soil-moisture LSTM work, for context: Kim, Lee, Chung et al. (2026),
[*Development of long short-term memory models using rainfall and soil moisture to predict soil moisture dynamics*](https://journals.plos.org/plosone/article?id=10.1371%2Fjournal.pone.0353268),
PLOS ONE — the closest setup (hourly in-situ soil moisture plus rainfall, MinMax scaling, chronological splits,
grid search); errors grew with forecast horizon.

In [ ]:
class SimpleLSTM(nn.Module):
    def __init__(self, num_features, hidden_size, num_layers):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=num_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, 1)               # one direction: no concatenation

    def forward(self, x):
        # x: (batch, 24, num_features)
        _, (h_n, _) = self.lstm(x)          # h_n: (num_layers, batch, hidden_size); h0/c0 default to zeros
        return self.fc(h_n[-1])             # last layer, after reading hour 1 -> 24  -> (batch, 1)


class SimpleBiLSTM(nn.Module):
    def __init__(self, num_features, hidden_size, num_layers):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=num_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
        )
        self.fc = nn.Linear(2 * hidden_size, 1)           # 2x: forward + backward summaries concatenated

    def forward(self, x):
        # x: (batch, 24, num_features)
        _, (h_n, _) = self.lstm(x)          # h_n: (num_layers * 2, batch, hidden_size); h0/c0 default to zeros
        h_forward = h_n[-2]                 # last layer, forward direction  (read hour 1 -> 24)
        h_backward = h_n[-1]                # last layer, backward direction (read hour 24 -> 1)
        h = torch.cat([h_forward, h_backward], dim=1)   # -> (batch, 2 * hidden_size)
        return self.fc(h)                   # -> (batch, 1)


MODEL_CLASSES = {"lstm": SimpleLSTM, "bilstm": SimpleBiLSTM}
MODEL_LABELS = {"lstm": "LSTM", "bilstm": "BiLSTM"}


def build_model(config, num_features):
    """The model named by config["model_type"], sized by config["hidden_size"] and config["num_layers"]."""
    return MODEL_CLASSES[config["model_type"]](num_features, hidden_size=config["hidden_size"],
                                               num_layers=config["num_layers"])


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


# Same interface: (batch, 24, F) in, (batch, 1) out
for model_type in MODEL_CLASSES:
    m = build_model({"model_type": model_type, "num_layers": 1, "hidden_size": 32}, len(FEATURES))
    assert m(torch.zeros(5, WINDOW_SIZE, len(FEATURES))).shape == (5, 1)
    print(f"{MODEL_LABELS[model_type]:6s} (1 layer, hidden 32, {len(FEATURES)} inputs): "
          f"{count_parameters(m):,} parameters")

## 7. Evaluation metrics

MSE, RMSE (√MSE, in `SWC_20` units — the score used for retuning), MAE, SMAPE, RSE and
CORR, written from their definitions and sanity-checked on known cases. MAPE is included for reference only:
it divides by the actual value, and although `SWC_20` (roughly 0.14–0.5) is never near zero, SMAPE is the
more robust percentage error.

In [ ]:
def compute_metrics(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=np.float64).flatten()
    y_pred = np.asarray(y_pred, dtype=np.float64).flatten()

    mse = np.mean((y_true - y_pred) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_true - y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + eps))) * 100
    smape = np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + eps)) * 100
    rse = np.sum((y_pred - y_true) ** 2) / (np.sum((y_true - y_true.mean()) ** 2) + eps)
    corr = np.corrcoef(y_true, y_pred)[0, 1] if min(np.std(y_true), np.std(y_pred)) > 1e-12 else float("nan")

    return {"MSE": mse, "RMSE": rmse, "MAE": mae, "MAPE": mape, "SMAPE": smape, "RSE": rse, "CORR": corr}


# Sanity checks against known cases
perfect = compute_metrics([1, 2, 3], [1, 2, 3])
assert perfect["MSE"] == 0 and perfect["MAE"] == 0 and abs(perfect["CORR"] - 1.0) < 1e-9
assert abs(compute_metrics([1.0], [0.0])["MAPE"] - 100.0) < 1e-6
assert abs(compute_metrics([0.0, 0.0], [3.0, 4.0])["RMSE"] - np.sqrt(12.5)) < 1e-12

## 8. Training loop and training helpers

A plain, manual PyTorch loop: Adam, MSE, early stopping on the validation loss with the best epoch's weights
restored. `train_model` returns the per-epoch training and validation losses, so the number of epochs run and the
best epoch are always known. `run_final` trains one model on the training years (early-stopped on the validation
year) and predicts every observed-target window of train, validation and test, in original `SWC_20` units.
Validation losses and predictions are computed in batches of `EVAL_BATCH_SIZE` (1024), which only affects speed;
training batches stay at `batch_size` (16).

Each fit's windows are moved to the device once. `DeviceLoader` draws the shuffled sample order from an
index-only `DataLoader` with the same seeded generator and slices every batch from the device tensors, so the
batches are exactly those of a per-sample `DataLoader` without collating and copying each one; the training
loss is summed on the device and read once per epoch instead of after every step.

In [ ]:
def train_model(model, train_loader, val_loader, epochs=EPOCHS, patience=PATIENCE, lr=1e-3):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    train_losses, val_losses = [], []
    best_val_loss = float("inf")
    best_state = None
    best_epoch = 0
    epochs_without_improvement = 0

    for epoch in range(epochs):
        model.train()                          # training mode
        running_loss = 0.0
        for xb, yb in train_loader:            # batches already live on DEVICE
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.detach() * xb.size(0)   # summed on the device: no CPU-GPU sync every step
        train_loss = running_loss.item() / len(train_loader.dataset)
        train_losses.append(train_loss)

        model.eval()                           # evaluation mode
        val_running = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                pred = model(xb)
                val_running += criterion(pred, yb).item() * xb.size(0)
        val_loss = val_running / len(val_loader.dataset)
        val_losses.append(val_loss)

        if val_loss < best_val_loss - 1e-7:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            best_epoch = epoch + 1
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)      # restore the best-validation epoch

    return model, train_losses, val_losses, best_epoch


class DeviceLoader:
    """(X, y) batches sliced from tensors that live on DEVICE. The sample order comes from an index-only
    DataLoader with the same generator, so the batches are exactly those of DataLoader(TensorDataset(X, y), ...)
    -- without collating every sample on the CPU and copying each batch to the GPU."""
    def __init__(self, X, y, batch_size, shuffle=False, generator=None):
        self.X = torch.as_tensor(X, device=DEVICE)
        self.y = torch.as_tensor(y, device=DEVICE).unsqueeze(1)
        self.dataset = self.y                  # len(loader.dataset) = number of samples, as for a DataLoader
        self.batch_size = batch_size
        self.index_loader = DataLoader(torch.arange(len(self.y)), batch_size=batch_size, shuffle=shuffle,
                                       generator=generator)

    def __iter__(self):
        order = torch.cat(list(self.index_loader)).to(DEVICE)   # this epoch's sample order, one copy to DEVICE
        for start in range(0, len(order), self.batch_size):
            idx = order[start:start + self.batch_size]
            yield self.X[idx], self.y[idx]


def make_loaders(X_train, y_train, X_val, y_val, batch_size, seed=SEED):
    g = torch.Generator().manual_seed(seed)
    return (DeviceLoader(X_train, y_train, batch_size, shuffle=True, generator=g),
            DeviceLoader(X_val, y_val, EVAL_BATCH_SIZE))


def predict(model, X, batch_size=EVAL_BATCH_SIZE):
    """Raw network outputs (scaled) for X, in order."""
    model.eval()
    with torch.no_grad():
        return torch.cat([model(torch.tensor(X[i:i + batch_size]).to(DEVICE)).cpu()
                          for i in range(0, len(X), batch_size)]).numpy().flatten()


def run_final(feature_cols, config, epochs=EPOCHS, patience=PATIENCE, seed=SEED):
    """Trains one model on the training years (early stopping on the validation year) and predicts every
    observed-target window of train, val and test. Predictions and targets are in original SWC_20 units."""
    t0 = time.time()
    splits = {"train": (train_df, train_scaled), "val": (val_df, val_scaled), "test": (test_df, test_scaled)}
    windows = {name: observed_windows(part, part_s, feature_cols) for name, (part, part_s) in splits.items()}
    train_loader, val_loader = make_loaders(*windows["train"][:2], *windows["val"][:2], config["batch_size"], seed)

    torch.manual_seed(seed)
    model = build_model(config, len(feature_cols))
    model, train_losses, val_losses, best_epoch = train_model(model, train_loader, val_loader, epochs=epochs,
                                                              patience=patience, lr=config["lr"])
    result = {"features": list(feature_cols), "config": config, "train_losses": train_losses,
              "val_losses": val_losses, "best_epoch": best_epoch, "epochs_run": len(train_losses)}
    for name, (part, _) in splits.items():
        X, _, end_idx, target_idx = windows[name]
        result[name] = {"y_true": part[TARGET_COL].to_numpy()[target_idx],      # observed values, unscaled
                        "y_pred": to_level(X, predict(model, X), feature_cols, scaler),
                        "end_time": part.index[end_idx], "target_time": part.index[target_idx]}
    result["runtime_sec"] = time.time() - t0
    return result

## 9. Naive baselines: last value and moving average

Two models with nothing to learn, scored on **exactly the same targets** as the LSTM (same windows, same horizon,
same observed-target filter):

- **Last value (persistence):** `SWC_20` at the target time equals `SWC_20` at the window's last hour.
- **Moving average (24h):** the mean `SWC_20` over the 24-hour input window.

Computed from the **unscaled** partitions, for train, validation and test (the training-set comparison is what
tells "learning the task" apart from "memorizing persistence" in Section 13), and for each CV fold's validation
years (Section 10).

In [ ]:
def naive_baselines(data_df, ma_window=WINDOW_SIZE):
    """True targets and naive predictions for every window with an observed target, in original SWC_20 units."""
    X, y, _, target_idx = make_windows(data_df, [TARGET_COL], TARGET_COL, WINDOW_SIZE, HORIZON)
    keep = observed_target_mask(data_df, target_idx)   # same targets as the LSTM
    swc = X[keep, :, 0]
    return y[keep], {
        "Last value": swc[:, -1],
        f"Moving average ({ma_window}h)": swc[:, -ma_window:].mean(axis=1),
    }


naive = {name: naive_baselines(part) for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]}
NAIVE_NAMES = list(naive["test"][1])

## 10. Whole-year cross-validation folds

Retuning is scored **inside the training years (2015–2022) only**, on expanding-window
folds made of whole calendar years (`CV_FOLD_YEARS`). Each fold trains on the past and validates on the two years
right after it:

| fold | train | validate |
|---|---|---|
| 1 | 2015–2018 (4 years) | 2019–2020 |
| 2 | 2015–2020 (6 years) | 2021–2022 |

Every training part spans at least two complete years, so the model sees the seasonal cycle repeat, and every
validation part is complete years, so seasonal performance is never judged on a partial cycle.

**Inside each fold:** a new `MinMaxScaler` is fit on the fold's training part only; windows are built separately
for each part; non-observed targets are dropped; the model trains for up to `TUNING_EPOCHS` (15) epochs with
patience `CV_PATIENCE` (3); and the fold's score is the RMSE of its validation predictions in original `SWC_20` units. A
configuration is scored by its **mean validation RMSE across the folds**. Each fold's last-value
(persistence) RMSE on the same validation windows is reported next to every score, so a configuration can be
read against persistence on that fold, not only against the other configurations.

The shorter CV early stopping is this version's main saving. With patience 10 every fit runs at least 11 epochs,
and in scratch runs with the change target the best epoch was 1, so ten of those epochs were wasted; the price
is that patience 3 can occasionally stop a fit before a later, better epoch (validation loss is noisy). The
final model keeps patience 10.

The fits are independent and each is seeded identically, so `cv_scores` runs them all in one batch: in parallel
worker processes when `N_JOBS` > 1 (CPU), one at a time in this process when `N_JOBS` = 1 (GPU).

In [ ]:
# Everything a CV fit reads: the model inputs and the target's origin (as a category, to keep worker inputs small)
CV_COLS = FEATURES + [f"{TARGET_COL}_Origin"]
cv_folds, FOLD_LAST_VALUE_RMSE = [], []
for k, (fold_train_years, fold_val_years) in enumerate(CV_FOLD_YEARS, start=1):
    assert set(fold_train_years) | set(fold_val_years) <= set(TRAIN_YEARS), "CV must stay inside the training years"
    assert max(fold_train_years) < min(fold_val_years), "always train on the past, validate on the future"
    assert len(fold_train_years) >= 2 and len(fold_val_years) >= 1
    tr, va = (rows_in_years(train_df, years)[CV_COLS].astype({f"{TARGET_COL}_Origin": "category"})
              for years in (fold_train_years, fold_val_years))
    dropped = {}
    for part_name, part, years in (("train", tr, fold_train_years), ("val", va, fold_val_years)):
        assert_whole_years(part, years)
        _, _, end_idx, target_idx = make_windows(part, [TARGET_COL], TARGET_COL, WINDOW_SIZE, HORIZON)
        check_window_alignment(part.index, end_idx, target_idx, WINDOW_SIZE, HORIZON)   # windows inside each part
        dropped[part_name] = int((~observed_target_mask(part, target_idx)).sum())
    y_true, naive_preds = naive_baselines(va)   # persistence on exactly this fold's scored validation windows
    FOLD_LAST_VALUE_RMSE.append(float(compute_metrics(y_true, naive_preds["Last value"])["RMSE"]))
    cv_folds.append((tr, va))
    print(f"fold {k}: train {year_span(fold_train_years)} ({len(fold_train_years)} years, {len(tr):5d} rows) | "
          f"validate {year_span(fold_val_years)} ({len(fold_val_years)} years, {len(va):5d} rows, "
          f"{len(y_true)} scored windows) | last-value RMSE {FOLD_LAST_VALUE_RMSE[-1]:.6f} | "
          f"non-observed targets dropped: {dropped}")

In [ ]:
def fit_fold(config, feature_cols, k, epochs=TUNING_EPOCHS):
    """One CV fit: trains `config` with `feature_cols` on fold k's training years, early-stopped on its validation
    years. Returns (validation RMSE in original SWC_20 units on observed targets, best epoch, epochs run, seconds).
    """
    t0 = time.time()
    if N_JOBS > 1:
        torch.set_num_threads(1)       # one thread per worker process; at batch size 16 more threads do not help
    fold_train, fold_val = cv_folds[k]
    scale_cols = list(dict.fromkeys(list(feature_cols) + [TARGET_COL]))   # inputs + target, no duplicates
    fold_scaler = MinMaxScaler().fit(fold_train[scale_cols])            # fit on this fold's training part only
    Xtr, ytr, _, _ = observed_windows(fold_train, scale(fold_train, scale_cols, fold_scaler), feature_cols)
    Xv, yv, _, va_tgt = observed_windows(fold_val, scale(fold_val, scale_cols, fold_scaler), feature_cols)
    train_loader, val_loader = make_loaders(Xtr, ytr, Xv, yv, config["batch_size"])

    torch.manual_seed(SEED)
    model = build_model(config, len(feature_cols))
    model, train_losses, _, best_epoch = train_model(model, train_loader, val_loader, epochs=epochs,
                                                     patience=CV_PATIENCE, lr=config["lr"])
    # RMSE of the restored best-epoch model, in original SWC_20 units
    pred = to_level(Xv, predict(model, Xv), feature_cols, fold_scaler)
    y_true = fold_val[TARGET_COL].to_numpy()[va_tgt]                    # observed values, unscaled
    return float(compute_metrics(y_true, pred)["RMSE"]), best_epoch, len(train_losses), time.time() - t0


_cv_cache = {}   # (features, config) -> CV result, so an identical evaluation is never re-run


def cv_key(config, feature_cols):
    return tuple(feature_cols), tuple(sorted(config.items()))


def cv_scores(jobs, epochs=TUNING_EPOCHS):
    """CV results for a list of (config, feature_cols) jobs, in order. Each result is a dict: mean validation
    RMSE across folds, per-fold RMSEs, best epochs and epochs run, and the summed fit time. Every uncached
    (job, fold) fit runs in one batch over N_JOBS processes."""
    todo = {}
    for config, cols in jobs:
        key = cv_key(config, cols)
        if key not in _cv_cache:
            todo[key] = (config, list(cols))
    n_folds = len(cv_folds)
    fits = Parallel(n_jobs=N_JOBS)(delayed(fit_fold)(config, cols, k, epochs)
                                   for config, cols in todo.values() for k in range(n_folds))
    for i, key in enumerate(todo):
        rmses, best_epochs, epochs_run, seconds = map(list, zip(*fits[i * n_folds:(i + 1) * n_folds]))
        _cv_cache[key] = {"mean": float(np.mean(rmses)), "folds": rmses, "best_epochs": best_epochs,
                          "epochs_run": epochs_run, "fit_sec": float(np.sum(seconds))}
    return [_cv_cache[cv_key(config, cols)] for config, cols in jobs]


def fold_columns(result):
    """Table columns for one CV result, with each fold's last-value RMSE on the same validation windows."""
    return {"mean_val_RMSE": result["mean"], "std_val_RMSE": float(np.std(result["folds"])),
            "mean_last_value_RMSE": float(np.mean(FOLD_LAST_VALUE_RMSE)),
            **{f"fold{k}_RMSE": r for k, r in enumerate(result["folds"], start=1)},
            **{f"fold{k}_last_value_RMSE": r for k, r in enumerate(FOLD_LAST_VALUE_RMSE, start=1)},
            **{f"fold{k}_best_epoch": e for k, e in enumerate(result["best_epochs"], start=1)},
            **{f"fold{k}_epochs_run": e for k, e in enumerate(result["epochs_run"], start=1)}}


def fmt_folds(values):
    return " / ".join(f"{v:.6f}" for v in values)

## 11. Retuning: LSTM vs BiLSTM

The 16-combination grid (`ParameterGrid(PARAM_GRID)`: `model_type` {lstm, bilstm} × `num_layers` {1, 2} ×
`hidden_size` {32, 48} × `lr` {0.001, 0.005}, `batch_size` 16) is scored on the whole-year folds with the two fixed
inputs; the lowest mean CV RMSE gives the final configuration, including whether the final model is
unidirectional or bidirectional. **The test year is never touched here.**

The full table lists every configuration with its per-fold RMSEs and fold std, the folds' last-value RMSEs, its gap
to the best mean, and on how many folds it is worse than the best configuration: a runner-up that is worse on only
some folds, or by much less than the fold std, is not clearly distinguishable from the winner. The second table puts
LSTM and BiLSTM side by side for each remaining setting.

In [ ]:
def grid_search_cv(feature_cols, param_grid=PARAM_GRID):
    """Scores every ParameterGrid combination on the same CV folds. Returns (table sorted by mean validation
    RMSE, best config)."""
    grid = list(ParameterGrid(param_grid))
    print(f"--- Grid search: {len(grid)} configs, {HORIZON}h horizon, features={list(feature_cols)}, "
          f"{len(cv_folds)} folds, up to {TUNING_EPOCHS} epochs (patience {CV_PATIENCE}), N_JOBS={N_JOBS} ---")
    t0 = time.time()
    results = cv_scores([(config, list(feature_cols)) for config in grid])
    rows = [{**{k: config[k] for k in param_grid}, "n_params": count_parameters(build_model(config, len(feature_cols))),
             **fold_columns(r)} for config, r in zip(grid, results)]
    best_row = min(rows, key=lambda r: r["mean_val_RMSE"])
    best_config = {k: best_row[k] for k in param_grid}

    table = pd.DataFrame(rows).sort_values("mean_val_RMSE").reset_index(drop=True)
    fold_cols = [f"fold{k}_RMSE" for k in range(1, len(cv_folds) + 1)]
    at = table.columns.get_loc("std_val_RMSE") + 1
    table.insert(at, "gap_to_best", table["mean_val_RMSE"] - best_row["mean_val_RMSE"])
    table.insert(at + 1, "folds_worse_than_best",
                 (table[fold_cols].to_numpy() > np.array([best_row[c] for c in fold_cols])).sum(axis=1))
    print(f"  {time.time() - t0:.0f}s; best config: {best_config} (mean val RMSE={best_row['mean_val_RMSE']:.6f}, "
          f"folds {fmt_folds([best_row[c] for c in fold_cols])})")
    return table, best_config


retune_grid, final_config = grid_search_cv(FEATURES)
retune_grid.to_csv(output_path("retune_grid_search.csv"), index=False)
display(retune_grid)

# LSTM and BiLSTM side by side for each remaining setting, and the best of each
side_by_side = retune_grid.pivot_table(index=["num_layers", "hidden_size", "lr"], columns="model_type",
                                       values=["mean_val_RMSE", "std_val_RMSE", "n_params"], aggfunc="first")
side_by_side[("bilstm_minus_lstm", "")] = (side_by_side[("mean_val_RMSE", "bilstm")]
                                          - side_by_side[("mean_val_RMSE", "lstm")])
display(side_by_side)
for model_type, group in retune_grid.groupby("model_type"):
    top = group.iloc[0]   # retune_grid is sorted by mean CV RMSE
    print(f"best {MODEL_LABELS[model_type]:6s}: mean CV RMSE {top['mean_val_RMSE']:.6f}, folds "
          f"{fmt_folds([top[f'fold{k}_RMSE'] for k in range(1, len(cv_folds) + 1)])} "
          f"(layers {top['num_layers']}, hidden {top['hidden_size']}, lr {top['lr']}, {top['n_params']:,} parameters)")

## 12. Final model

Trained on the training years 2015–2022 (scaler fit there only) with the two inputs and the retuned
configuration — LSTM or BiLSTM, whichever the grid search picked — early-stopped on 2023 (up to 30 epochs,
patience 10, best-validation weights restored), then evaluated once on 2024.

In [ ]:
MODEL_NAME = f"{MODEL_LABELS[final_config['model_type']]} ({EXPERIMENT_NAME})"   # labels the tables and plots
print(f"{EXPERIMENT_NAME}: training the {HORIZON}h model, config {final_config}, features {FEATURES} ...")
final = run_final(FEATURES, final_config)
print(f"  epochs run: {final['epochs_run']} (max {EPOCHS}, patience {PATIENCE}); best epoch: {final['best_epoch']}; "
      f"{final['runtime_sec']:.0f}s")

# the model and the naive baselines must be scored on the identical targets
for name in ("train", "val", "test"):
    assert np.allclose(final[name]["y_true"], naive[name][0], atol=1e-6)

with open(output_path("tuning_summary.json"), "w") as f:
    json.dump({"experiment": EXPERIMENT_NAME, "horizon": HORIZON, "features": FEATURES, "train_years": TRAIN_YEARS,
               "val_years": VAL_YEARS, "test_years": TEST_YEARS, "cv_fold_years": CV_FOLD_YEARS,
               "ppt_deadband_mm": PPT_DEADBAND_MM, "predict_change": PREDICT_CHANGE,
               "final_config": final_config, "final_cv_RMSE": float(retune_grid["mean_val_RMSE"].iloc[0]),
               "final_best_epoch": final["best_epoch"], "final_epochs_run": final["epochs_run"],
               "max_epochs": EPOCHS, "patience": PATIENCE, "tuning_epochs": TUNING_EPOCHS,
               "cv_patience": CV_PATIENCE}, f, indent=2)

### 12.1 Training and validation loss per epoch

How to read the two curves: both falling and then levelling off together means the model is learning; training
loss still falling while validation loss rises means overfitting; both flat from the first epoch or two (with the
best epoch near the start) means a plateau — the model reached a low-loss solution almost immediately and further
epochs do not move it. Losses are MSE on the scaled target (with `PREDICT_CHANGE`, the scaled change, whose squared
error equals that of the scaled level).

In [ ]:
epochs_axis = np.arange(1, final["epochs_run"] + 1)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(epochs_axis, final["train_losses"], color=C_MODEL, linewidth=2, marker="o", markersize=4,
        label="training loss")
ax.plot(epochs_axis, final["val_losses"], color=C_LAST, linewidth=2, marker="o", markersize=4,
        label="validation loss")
ax.axvline(final["best_epoch"], color=C_MUTED, linestyle=":", linewidth=1.2,
           label=f"best epoch ({final['best_epoch']}), weights restored")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE (scaled SWC_20)")
ax.set_xticks(epochs_axis)
ax.set_title(f"{MODEL_NAME}, {HORIZON}h: loss per epoch, {final['epochs_run']} epochs run "
             f"(max {EPOCHS}, patience {PATIENCE})")
ax.grid(alpha=0.3, linewidth=0.5)
ax.legend()
plt.tight_layout()
plt.show()

## 13. Evaluation

### 13.1 Test metrics and the persistence check

The first table compares the model with the naive baselines on the test year (2024). The second reports RMSE on
**every** partition against the same baselines on identical targets; `skill_vs_last_value` = 1 − RMSE(model) /
RMSE(last value), positive when the model beats persistence. Beating persistence on training data but not on
validation/test indicates overfitting; failing to beat it even on training data means the model is not learning
the task.

In [ ]:
METRIC_COLS = ["MSE", "RMSE", "MAE", "MAPE", "SMAPE", "RSE", "CORR"]

y_true_test, naive_test = naive["test"]
test_table = pd.DataFrame({MODEL_NAME: compute_metrics(final["test"]["y_true"], final["test"]["y_pred"]),
                           **{name: compute_metrics(y_true_test, pred) for name, pred in naive_test.items()}}
                          ).T[METRIC_COLS]
test_table.index.name = f"{HORIZON}h horizon, test {year_span(TEST_YEARS)}"
test_table.to_csv(output_path("test_metrics.csv"))
display(test_table)
print(f"Features: {FEATURES}\nConfig:   {final_config}")

persistence_rows = {}
for name in ("train", "val", "test"):
    y_true, preds = naive[name]
    rmse_model = compute_metrics(final[name]["y_true"], final[name]["y_pred"])["RMSE"]
    rmse_last = compute_metrics(y_true, preds["Last value"])["RMSE"]
    persistence_rows[name] = {"model_RMSE": rmse_model, "last_value_RMSE": rmse_last,
                              "moving_average_RMSE": compute_metrics(y_true, preds[NAIVE_NAMES[1]])["RMSE"],
                              "skill_vs_last_value": 1 - rmse_model / rmse_last, "n_targets": len(y_true)}
persistence_table = pd.DataFrame(persistence_rows).T
persistence_table.index.name = "partition"
display(persistence_table.astype({"n_targets": int}).round(5))

### 13.2 Alignment check on real timestamps

A handful of test windows, evenly spaced through the test year: the input window, the target timestamp, the
prediction, the actual `SWC_20` looked up **by that timestamp** in the full dataframe, and the last-value
baseline. The assert checks the same timestamp lookup for every scored test target, so the plots below draw each
prediction at the hour it is a forecast for.

In [ ]:
t = final["test"]
assert np.allclose(t["y_true"], df.loc[t["target_time"], TARGET_COL].to_numpy())
assert ((t["target_time"] - t["end_time"]) == pd.Timedelta(hours=HORIZON)).all()
pick = np.linspace(0, len(t["y_pred"]) - 1, 8).astype(int)
display(pd.DataFrame({
    "input_start": t["end_time"][pick] - pd.Timedelta(hours=WINDOW_SIZE - 1),
    "input_end": t["end_time"][pick],
    "target_time": t["target_time"][pick],
    "predicted": t["y_pred"][pick],
    "actual": df.loc[t["target_time"][pick], TARGET_COL].to_numpy(),
    "last_value": naive_test["Last value"][pick],
}).round(4))

### 13.3 Test year by month

One panel per calendar month of the test year, by target time (each panel's y-axis is scaled to that month so
small errors and lags stay visible). The last-value baseline is the actual series shifted right by exactly
48 hours; where the model line sits on top of it, the model is effectively predicting "no change". The moving
average baseline is in the tables above.

In [ ]:
model_label = MODEL_LABELS[final_config["model_type"]]
monthly = pd.DataFrame({"actual": t["y_true"], "model": t["y_pred"], "last_value": naive_test["Last value"]},
                       index=t["target_time"])
months = monthly.index.to_period("M").unique()
ncols = 3
nrows = int(np.ceil(len(months) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 2.7 * nrows), squeeze=False)
for ax, month in zip(axes.flat, months):
    m = monthly[monthly.index.to_period("M") == month]
    ax.plot(m.index, m["actual"], color=C_ACTUAL, linewidth=1.4, label="Actual")
    ax.plot(m.index, m["model"], color=C_MODEL, linewidth=1.2, label=MODEL_NAME)
    ax.plot(m.index, m["last_value"], color=C_LAST, linewidth=1.0, linestyle="--",
            label=f"Last value (actual shifted +{HORIZON}h)")
    rmse_model = compute_metrics(m["actual"], m["model"])["RMSE"]
    rmse_last = compute_metrics(m["actual"], m["last_value"])["RMSE"]
    ax.set_title(f"{month.strftime('%b %Y')}   RMSE: {model_label} {rmse_model:.4f} | last value {rmse_last:.4f}",
                 fontsize=9)
    ax.set_xlim(m.index.min(), m.index.max())
    ax.xaxis.set_major_locator(mdates.DayLocator(bymonthday=[1, 8, 15, 22, 29]))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d"))
    ax.tick_params(labelsize=8)
    ax.grid(alpha=0.3, linewidth=0.5)
for ax in axes.flat[len(months):]:
    ax.set_visible(False)
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.0))
fig.suptitle(f"{STATION}: SWC_20 {HORIZON}h ahead, test year {year_span(TEST_YEARS)} by month "
             f"(x = target time, day of month)", y=1.025)
fig.supylabel("SWC_20")
plt.tight_layout(rect=(0, 0, 1, 0.975))
plt.show()